# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [25]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [26]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [27]:
w1 = torch.tensor(0., requires_grad=True)
w2 = torch.tensor(0., requires_grad=True)
b = torch.tensor(0.,requires_grad=True)

optimizer = torch.optim.SGD([w1,w2,b],lr=0.1)

for epoch in range(1,2001):
    optimizer.param_groups[0]['lr'] = 0.1/(epoch**0.5)
    optimizer.zero_grad()
    y_pred = predict(x1,x2,w1,w2,b)
    per_sample_loss = torch.abs(y_pred -y)
    loss = per_sample_loss.mean()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    y_pred = predict(x1,x2,w1,w2,b)
    per_sample_loss = torch.abs(y_pred - y)
    loss = per_sample_loss.mean()

print('w1:', w1, 'shape:', tuple(w1.shape))
print('w2:', w2, 'shape:', tuple(w2.shape))
print('b: ', b,  'shape:', tuple(b.shape))

print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y,  'shape:', tuple(y.shape))

print('y_pred:', y_pred, 'shape:', tuple(y_pred.shape))
print('per_sample_loss:', per_sample_loss, 'shape:', tuple(per_sample_loss.shape))
print('loss:', loss, 'shape:', tuple(loss.shape))

w1: tensor(1.9990, requires_grad=True) shape: ()
w2: tensor(-3.0010, requires_grad=True) shape: ()
b:  tensor(0.9994, requires_grad=True) shape: ()
x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)
y_pred: tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954]) shape: (6,)
per_sample_loss: tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046]) shape: (6,)
loss: tensor(0.0024) shape: ()


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

## 我的观察与实验

1. **现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）**
   - 一次预测需要**两个**输入张量（`x1` 和 `x2`）。
   - `x1` 和 `x2` 的 shape 都是 `(6,)`，表示共有 6 个样本。`x1[i]` 和 `x2[i]` 共同构成第 `i` 个样本的两个特征，二者按样本索引一一对应。

2. **对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）**
   - **预测公式**：由单个输入变为使用两个输入：`y_pred = w1 * x1 + w2 * x2 + b`。
   - **参数列表**：参数从原来的两个变成了三个（`w1`、`w2`、`b`），并且都被交给了优化器。
   - **损失计算**：从原来的平方损失改为了绝对值损失，先计算逐样本误差 `torch.abs(y_pred - y)`，再用 `.mean()` 得到标量平均损失。
   - **优化器的基本步骤没有变**：依然是“清空梯度 -> 前向预测 -> 计算损失 -> 反向传播 -> 更新参数”这五步循环。

3. **记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）**
   - **最终参数**：训练后参数应非常接近 `w1 ≈ 2.0`, `w2 ≈ -3.0`, `b ≈ 1.0`。（注：根据数据可推导出真实线性关系为 `y = 2*x1 - 3*x2 + 1`）。
   - **预测与损失**：`y_pred` 应非常接近真实值 `[-6, -1, -5, 6, -4, 4]`，平均损失（MAE）应非常接近 `0`。
   - **如何判断训练有效**：一是看平均损失是否随着轮次增加显著下降并趋近于 0；二是看最终预测值 `y_pred` 是否与目标值 `y` 高度吻合；三是看参数是否稳定在我们预期的理论值附近。如果损失不再下降且预测准确，则说明训练有效。

4. **为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）**
   - 因为模型是线性的，虽然初始预测值为 0，但真实目标 `y` 不全为 0，所以存在预测误差。
   - 对于绝对值损失，误差项 `y_pred - y` 的符号会直接决定梯度的方向（次梯度不为 0）。因为误差不为 0，参数 `w1`、`w2`、`b` 就会获得非零的梯度，从而能够离开 0 开始更新。
   - 上周的例子如果初始化为 0，可能由于平方损失的导数在 0 处刚好也是 0，导致“梯度消失”而停滞；而本次是线性模型+绝对值损失，不存在该问题。
